# Membrete

| | |
|---|---|
| **Autor** | Avila Larrotta |
| **Programa** | Maestría en Inteligencia Artificial |
| **Asignatura** | Computación de Alto Rendimiento (HPC) |
| **Trabajo** | Predicción de ingresos superiores a 50 000 USD (Census Income) con Spark MLlib sobre Hadoop HDFS |
| **Entorno** | Clúster del curso: JupyterHub `10.195.34.34:8000`, Spark + Hadoop HDFS |
| **Fecha de entrega** | 22/11/2026 |

# Contexto

El conjunto **Census Income** (también llamado *Adult*) proviene del Repositorio de Aprendizaje Automático de
la UCI. Ronny Kohavi y Barry Becker lo extrajeron de la base del **censo de Estados Unidos de 1994**. Cada
registro es una persona mayor de 16 años con datos demográficos (edad, sexo, raza, país de origen), educativos
(nivel y años de estudio), familiares (estado civil, parentesco) y laborales (clase de trabajo, ocupación,
horas por semana, ganancias y pérdidas de capital).

La tarea es de **clasificación binaria supervisada**: predecir si los ingresos anuales de una persona
**superan los 50 000 USD** (`>50K`) o no (`<=50K`). Un modelo así tiene usos como focalizar programas sociales,
segmentar mercados o estudiar qué factores socioeconómicos se asocian a mayores ingresos.

**Dataset.** La versión de UCI trae dos archivos: `adult.data` (32 561 registros) y `adult.test`
(16 281 registros). Ambos tienen 14 variables explicativas y la etiqueta `income`. Los valores faltantes vienen
marcados con `?`, el archivo no tiene encabezado y en `adult.test` la etiqueta termina en punto (`>50K.`).
Este cuaderno lee los archivos desde **Hadoop HDFS** y une ambos para tener ≈ 48 842 registros.

**Entorno HPC.** Todo el procesamiento se hace con **Apache Spark (PySpark)**: lectura distribuida desde HDFS,
limpieza, agregaciones del análisis exploratorio, codificación, escalado, entrenamiento y evaluación con
**MLlib** (`pyspark.ml`). A pandas solo se pasan resultados agregados o columnas numéricas para graficar.

**Retos del problema.**
- Las clases están **desbalanceadas**: solo ≈ 24 % de las personas gana más de 50K.
- Hay valores faltantes en variables categóricas (clase de trabajo, ocupación y país).
- Hay variables con muchas categorías (ocupación, educación, país) y variables numéricas muy sesgadas
  (ganancia y pérdida de capital, casi siempre 0).

### Formulación del problema de aprendizaje (Tom Mitchell)

Según la definición de Mitchell, un programa *aprende* de la experiencia **E** respecto a una tarea **T** y una
medida de desempeño **P** si su desempeño en T, medido con P, mejora con E. En este trabajo:

- **T (tarea):** clasificar si los ingresos anuales de una persona superan los 50 000 USD.
- **E (experiencia):** ≈ 48 mil registros del censo de EE. UU. de 1994 leídos desde HDFS, de los cuales ≈ 80 % se usan para entrenar.
- **P (desempeño):** AUC-ROC, AUC-PR y F1 de la clase `>50K` sobre el conjunto de prueba.

Es un enfoque de **Software 2.0**: en lugar de programar reglas a mano (Software 1.0), la regla de decisión
se **aprende de los datos**. Se elige una hipótesis $h_\theta$ dentro de un espacio de hipótesis minimizando
una función de costo sobre la matriz de características $X \in \mathbb{R}^{n \times m}$ y el vector de
etiquetas $y$. Es aprendizaje **supervisado** y el modelo tiene un uso **predictivo** y también
**interpretativo** (sección 4.4).

# Objetivos

### Objetivo general
Construir y evaluar, con la metodología **CRISP-DM**, PySpark y Hadoop HDFS, al menos cinco modelos de **MLlib**
que predigan si los ingresos de una persona superan los 50 000 USD al año a partir de los datos del censo.

### Objetivos específicos
1. Cargar el dataset desde **Hadoop HDFS** y manipularlo con DataFrames distribuidos de Spark.
2. **Conocer los datos**: estructura, tipos, calidad (faltantes, duplicados, categorías) y relación de cada
   variable con el nivel de ingresos mediante análisis exploratorio.
3. **Preparar los datos**: limpieza, imputación, eliminación de variables redundantes, transformación de
   variables sesgadas, agrupación de categorías, codificación (StringIndexer + OneHotEncoder), ensamblado y
   escalado de características, y tratamiento del desbalance.
4. **Entrenar al menos cinco modelos de MLlib**: Regresión Logística, Árbol de Decisión, Random Forest,
   Gradient-Boosted Trees, Linear SVC, Perceptrón Multicapa y Naive Bayes.
5. **Evaluar y comparar** los modelos con accuracy, precisión, *recall* y F1 de la clase `>50K`,
   especificidad, AUC-ROC, AUC-PR y matriz de confusión.
6. Analizar los resultados, identificar los factores más asociados a ingresos altos y concluir cuál modelo
   conviene.
7. Aplicar con `pyspark.ml` otros temas del curso: validación cruzada *k-fold* y regularización
   Ridge/Lasso/Elastic Net, diagnóstico sesgo–varianza, suavizado de Laplace, del perceptrón al MLP,
   expansión polinómica como alternativa al kernel de la SVM, PCA y agrupamiento (K-Means, Bisecting K-Means
   y GMM).

# Desarrollo y documentación

El trabajo sigue las fases de **CRISP-DM**:

| Fase CRISP-DM | Sección del cuaderno |
|---|---|
| Comprensión del negocio | Contexto y Objetivos |
| Comprensión de los datos | 1. Conocimiento de datos |
| Preparación de los datos | 2. Preparación de datos |
| Modelado | 3. Modelos de MLlib |
| Evaluación | 4. Evaluación de modelos y 5. Análisis de resultados |
| Despliegue (recomendación) | 6. Conclusiones |

## 1. Conocimiento de datos
### 1.1 Bibliotecas y sesión de Spark

In [ ]:
#Importar bibliotecas generales
import numpy as np                               # Biblioteca para matemática y algebra lineal
import seaborn as sns                            # Biblioteca para estadística y gráficas
import matplotlib.pyplot as plt                  # Biblioteca que se usa para dar formato a las graficas
import pandas as pd                              # Biblioteca para generar objetos de tipo DataFrame (No big data)

In [ ]:
#Se importan las bibliotecas para PySpark
import findspark                                   # Permite cargar la ruta de Spark
findspark.init("/mnt/sda1/Cluster/Spark/")         # Ruta de spark lista

from pyspark import SparkConf, SparkContext        # Configuración global y punto de entrada para computación distribuida
from pyspark.sql import functions as F             # Funciones integradas para transformaciones, agregaciones y cálculo en columnas
from pyspark.sql import SQLContext, SparkSession   # SparkSession es el punto de entrada unificado moderno; SQLContext es la interfaz legacy
from pyspark.sql.types import *                    # Tipos de datos para definir esquemas estructurados de DataFrames

In [ ]:
configura = SparkConf().set("spark.ui.port","37001")                          # Puerto de la interfaz web (Spark UI) asignado en clase
configura.setAppName("AvilaSalarioSpark")                                     # Nombre de la aplicación, visible en la Spark UI

sparkAvila = SparkSession.builder.config(conf=configura).getOrCreate()        # Crea la sesión de Spark o reutiliza una existente
SQLContext(sparkContext=sparkAvila.sparkContext, sparkSession=sparkAvila)     # Contexto SQL legacy (compatibilidad)

print("Sesion creada por Avila para Census Income")
sparkAvila

### 1.2 Carga del dataset en Hadoop HDFS

Se crea la carpeta del taller en HDFS, se suben los archivos del censo (`adult.data` y `adult.test`, que
antes se cargan al directorio del cuaderno en JupyterHub) y se verifica que hayan quedado ahí.

In [ ]:
### Se consulta que hay dentro de Hadoop HDFS
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /

In [ ]:
### Se crea la carpeta del taller: FormatosHPC/Avila/TallerModelosClasificacion
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -mkdir -p /FormatosHPC/Avila/TallerModelosClasificacion

In [ ]:
### Se consulta que hay dentro de la carpeta FormatosHPC/Avila
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /FormatosHPC/Avila

In [ ]:
### Se suben a la carpeta /FormatosHPC/Avila/TallerModelosClasificacion los archivos del dataset
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -put -f adult.data /FormatosHPC/Avila/TallerModelosClasificacion
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -put -f adult.test /FormatosHPC/Avila/TallerModelosClasificacion

In [ ]:
### Se consulta que hay dentro de la carpeta TallerModelosClasificacion
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /FormatosHPC/Avila/TallerModelosClasificacion

Los archivos de UCI **no tienen encabezado**, separan con coma y un espacio, y `adult.test` empieza con una
línea de comentario (`|1x3 Cross validator`). Por eso se leen sin encabezado, se recortan los espacios y se
asignan los nombres de columna. Las filas cuya edad no es numérica (comentarios, líneas vacías o un encabezado
si se usara la versión `adult.csv` de Kaggle) se eliminan después.

In [ ]:
### Se requiere crear un objeto de tipo DataFrame con los archivos del censo
### Se carga el dataset desde la carpeta del taller en HDFS
urlAvila = ["hdfs://10.195.34.34:9000/FormatosHPC/Avila/TallerModelosClasificacion/adult.data",
            "hdfs://10.195.34.34:9000/FormatosHPC/Avila/TallerModelosClasificacion/adult.test"]

nombresOriginales = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
                     'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
                     'hours-per-week', 'native-country', 'income']

df00 = sparkAvila.read.format("csv") \
                 .option("header", "false") \
                 .option("ignoreLeadingWhiteSpace", "true") \
                 .option("ignoreTrailingWhiteSpace", "true") \
                 .load(urlAvila) \
                 .toDF(*nombresOriginales)

# Se ven los 5 primeros
df00.show(5)
print("Registros leídos:", df00.count())

### 1.3 Diccionario de datos

| Original | Nuevo nombre | Tipo | Descripción |
|---|---|---|---|
| age | Edad | Numérica | Edad en años |
| workclass | ClaseTrabajo | Categórica | Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked |
| fnlwgt | PesoMuestral | Numérica | Peso muestral del censo: cuántas personas representa el registro |
| education | Educacion | Categórica | Máximo nivel educativo alcanzado (16 niveles) |
| education-num | AniosEducacion | Numérica | Codificación ordinal del nivel educativo (1–16) |
| marital-status | EstadoCivil | Categórica | Married-civ-spouse, Never-married, Divorced, Separated, Widowed, … |
| occupation | Ocupacion | Categórica | Ocupación (14 categorías) |
| relationship | Parentesco | Categórica | Rol dentro del hogar: Husband, Wife, Own-child, Not-in-family, Unmarried, Other-relative |
| race | Raza | Categórica | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| sex | Sexo | Categórica | Male / Female |
| capital-gain | GananciaCapital | Numérica | Ganancias de capital (USD) |
| capital-loss | PerdidaCapital | Numérica | Pérdidas de capital (USD) |
| hours-per-week | HorasSemana | Numérica | Horas trabajadas por semana |
| native-country | PaisOrigen | Categórica | País de origen (41 países) |
| income | Ingreso | Binaria (objetivo) | `<=50K` → 0, `>50K` → 1 |

In [ ]:
# Nuevos nombres
nuevosNombres = ['Edad', 'ClaseTrabajo', 'PesoMuestral', 'Educacion', 'AniosEducacion', 'EstadoCivil',
                 'Ocupacion', 'Parentesco', 'Raza', 'Sexo', 'GananciaCapital', 'PerdidaCapital',
                 'HorasSemana', 'PaisOrigen', 'Ingreso']

# Se hace el cambio con nueva version
df01 = df00
for antes, nuevos in zip(df00.columns, nuevosNombres):
    df01 = df01.withColumnRenamed(antes, nuevos)

# Se eliminan las filas que no son datos (comentario de adult.test, líneas vacías o encabezados)
df01 = df01.where(F.col('Edad').rlike('^[0-9]+$'))

# Se verifica
print(df01.columns)
print("Registros válidos:", df01.count())

In [ ]:
#Se revisa el tipo de datos: todo llega como texto
df01.printSchema()

### Se requiere que
- `Edad`, `PesoMuestral`, `AniosEducacion`, `GananciaCapital`, `PerdidaCapital`, `HorasSemana`: **entero**.
- `Ingreso`: **entero binario** (0 = `<=50K`, 1 = `>50K`). Hay que quitar el punto final que trae `adult.test`.
- Las categóricas siguen como texto, y el símbolo `?` se convierte en **nulo** para poder contarlo.

In [ ]:
colEnteras = ['Edad', 'PesoMuestral', 'AniosEducacion', 'GananciaCapital', 'PerdidaCapital', 'HorasSemana']
colCategoricas = ['ClaseTrabajo', 'Educacion', 'EstadoCivil', 'Ocupacion', 'Parentesco', 'Raza', 'Sexo', 'PaisOrigen']

# Recordar cambio de version
df02 = df01
for c in colEnteras:
    df02 = df02.withColumn(c, F.col(c).cast('int'))

# '?' -> nulo en las categóricas
for c in colCategoricas:
    df02 = df02.withColumn(c, F.when(F.trim(F.col(c)) == '?', None).otherwise(F.trim(F.col(c))))

# Etiqueta: se quita el punto de adult.test y se convierte a 0/1
df02 = df02.withColumn('Ingreso', F.when(F.regexp_replace(F.trim(F.col('Ingreso')), r'\.', '') == '>50K', 1).otherwise(0))

## Se verifica
df02.show(5)
df02.printSchema()

### 1.4 Calidad de los datos: faltantes, duplicados y estadísticos

In [ ]:
# Cuantificar la cantidad de datos nulos (antes eran '?')
df02.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df02.columns]).show()

In [ ]:
print("Columnas:", len(df02.columns))
totalRegistros = df02.count()
print("Registros:", totalRegistros)

# Filas con al menos un faltante
filasNulas = df02.where(F.greatest(*[F.col(c).isNull().cast('int') for c in colCategoricas]) == 1).count()
print(f"Filas con al menos un faltante: {filasNulas} ({filasNulas/totalRegistros*100:.2f} %)")

# Registros duplicados
duplicados = totalRegistros - df02.dropDuplicates().count()
print(f"Registros duplicados: {duplicados}")

In [ ]:
# Estadísticos descriptivos de las variables numéricas (calculados en Spark)
df02.select(colEnteras).summary('count', 'mean', 'stddev', 'min', '25%', '50%', '75%', 'max').show()

In [ ]:
# Proporción de personas con ganancia o pérdida de capital igual a cero
df02.select(
    F.round(F.avg((F.col('GananciaCapital') == 0).cast('int')) * 100, 2).alias('% GananciaCapital = 0'),
    F.round(F.avg((F.col('PerdidaCapital') == 0).cast('int')) * 100, 2).alias('% PerdidaCapital = 0')
).show()

In [ ]:
# Se revisan las categorías de cada variable categórica
for c in colCategoricas:
    print(f"===== {c}: {df02.select(c).distinct().count()} categorías")
    df02.groupBy(c).count().orderBy(F.desc('count')).show(50, truncate=False)

In [ ]:
# Educacion y AniosEducacion: se verifica si son la misma información
df02.groupBy('Educacion', 'AniosEducacion').count().orderBy('AniosEducacion').show(20, False)

**Observaciones sobre la calidad de los datos**
- **Faltantes.** Solo hay en `ClaseTrabajo`, `Ocupacion` y `PaisOrigen`, es decir, los antiguos `?`. Las dos
  primeras casi siempre faltan juntas, porque son personas sin información laboral. En total afectan a un
  porcentaje pequeño de filas (≈ 7 %). Aun así, **no se eliminan**: se imputan como una categoría propia
  (ver 2.2), porque "no reportar el trabajo" puede ser informativo.
- **Duplicados.** Hay pocos registros idénticos. Con variables tan agregadas, dos personas distintas pueden
  coincidir en todo, pero las copias exactas no aportan información nueva y se eliminan.
- **Educacion vs AniosEducacion.** Cada nivel educativo corresponde exactamente a un valor de `AniosEducacion`
  (relación 1 a 1). Son la misma información, así que se conserva solo la versión numérica ordinal.
- **Capital.** Más del 90 % de las personas tiene ganancia y pérdida de capital iguales a 0, con unos pocos
  valores muy altos (la ganancia llega a 99 999, un valor tope del censo). Son variables muy sesgadas que hay
  que transformar.
- **PaisOrigen.** Hay 41 países, pero ≈ 90 % es `United-States` y la mayoría de los demás tiene menos de 1 %
  de los registros.
- **PesoMuestral (fnlwgt).** Es un peso de muestreo del censo y no una característica de la persona. Por eso
  no se usa como predictor.

### 1.5 Distribución de la variable objetivo

In [ ]:
### Se cuantifica cuántas personas ganan más y menos de 50K
distIngreso = df02.groupBy('Ingreso').count() \
                  .withColumn('Porcentaje', F.round(F.col('count') / F.lit(totalRegistros) * 100, 2)) \
                  .orderBy('Ingreso')
distIngreso.show()

pdfDist = distIngreso.toPandas()
plt.figure(figsize=(6,4))
sns.barplot(data=pdfDist, x='Ingreso', y='count')
plt.xticks([0, 1], ['<=50K', '>50K'])
plt.title("Distribución de la variable objetivo")
plt.ylabel("Cantidad")
plt.grid(axis='y')
plt.show()

**Observación.** Solo ≈ 24 % de las personas gana más de 50K, es decir, hay unas 3 personas de ingreso bajo
por cada una de ingreso alto. El desbalance es **moderado**. Un modelo que siempre responda `<=50K` tendría
≈ 76 % de exactitud sin ningún valor práctico, así que la evaluación se centra en las métricas de la clase
`>50K` y en el AUC, y en el entrenamiento se prueba el efecto de balancear las clases.

### 1.6 Análisis exploratorio: variables cuantitativas

Para graficar se pasan a pandas solo las columnas numéricas y la etiqueta.

In [ ]:
### Se convierten a pandas solo las columnas numéricas y la etiqueta
DataPandasN = df02.select(colEnteras + ['Ingreso']).toPandas()
DataPandasN['IngresoTxt'] = DataPandasN['Ingreso'].map({0: '<=50K', 1: '>50K'})

In [ ]:
## Histogramas de las variables numéricas
fig, ejes = plt.subplots(2, 3, figsize=(17, 8))
for eje, c in zip(ejes.ravel(), colEnteras):
    sns.histplot(data=DataPandasN, x=c, bins=50, kde=c in ['Edad', 'HorasSemana', 'AniosEducacion'], ax=eje)
    eje.set_title(f"Cantidad vs {c}")
    eje.set_ylabel("Cantidad")
    eje.grid()
plt.tight_layout()
plt.show()

In [ ]:
## Distribución de las numéricas según el nivel de ingreso
fig, ejes = plt.subplots(1, 3, figsize=(17, 4))
for eje, c in zip(ejes, ['Edad', 'AniosEducacion', 'HorasSemana']):
    sns.boxplot(data=DataPandasN, x='IngresoTxt', y=c, order=['<=50K', '>50K'], ax=eje)
    eje.set_title(f"{c} según Ingreso")
    eje.set_xlabel("Ingreso")
    eje.grid()
plt.tight_layout()
plt.show()

In [ ]:
## Distribución de la edad por nivel de ingreso (densidades normalizadas para comparar formas)
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=DataPandasN, x='Edad', hue='IngresoTxt', stat='density', common_norm=False, bins=40, ax=ejes[0])
ejes[0].set_title('Distribución de Edad por Ingreso'); ejes[0].grid()
sns.histplot(data=DataPandasN, x='HorasSemana', hue='IngresoTxt', stat='density', common_norm=False, bins=40, ax=ejes[1])
ejes[1].set_title('Distribución de HorasSemana por Ingreso'); ejes[1].grid()
plt.tight_layout()
plt.show()

In [ ]:
## Matriz de correlación de las variables numéricas con la etiqueta
plt.figure(figsize=(9, 6))
sns.heatmap(DataPandasN[colEnteras + ['Ingreso']].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlación de Pearson')
plt.show()

**Análisis de variables cuantitativas**
- **Edad:** está sesgada a la derecha (17 a 90 años, media ≈ 38–39). Las personas con ingresos `>50K` son en
  promedio mayores (mediana ≈ 44 frente a ≈ 34). El ingreso alto se concentra entre los 35 y 60 años y es raro
  en menores de 25. La relación no es lineal: sube con la edad y baja después de la jubilación.
- **AniosEducacion:** es la variable numérica que mejor separa las clases. La mayoría de las personas con
  `>50K` tiene 10 años de educación o más (bachillerato universitario en adelante). Por debajo de 9 casi nadie
  supera los 50K.
- **HorasSemana:** hay un pico muy marcado en 40 horas (jornada completa). Quienes ganan `>50K` trabajan más
  horas en promedio, y entre quienes trabajan menos de 30 horas los ingresos altos son escasos.
- **GananciaCapital / PerdidaCapital:** casi todos los valores son 0. Cuando no lo son, tienen mucho poder
  predictivo: tener ganancias de capital está muy asociado a `>50K`. Por su sesgo extremo se transforman con
  $\log(1+x)$.
- **PesoMuestral:** su correlación con el ingreso es prácticamente nula (≈ 0). Esto confirma que se puede
  descartar.
- **Correlaciones:** ninguna variable numérica está muy correlacionada con otra, así que no hay
  multicolinealidad. Las correlaciones con `Ingreso` más altas son las de `AniosEducacion`, `Edad`,
  `HorasSemana` y `GananciaCapital`.

### 1.7 Análisis exploratorio: variables cualitativas

Para cada variable categórica se calcula **en Spark** cuántas personas hay en cada categoría y qué porcentaje
de ellas gana `>50K`. Solo esa tabla resumida se pasa a pandas para graficar.

In [ ]:
def tasaPorCategoria(df, col, orden=None):
    """Agrega en Spark: cantidad de personas y % con Ingreso >50K por categoría, y lo grafica."""
    agregado = df.groupBy(col).agg(F.count('*').alias('Cantidad'),
                                   F.round(F.avg('Ingreso') * 100, 2).alias('Pct_>50K'))
    pdf = agregado.toPandas().fillna({col: 'Nulo'})
    pdf = pdf.set_index(col).loc[orden].reset_index() if orden else pdf.sort_values('Pct_>50K', ascending=False)

    fig, ejes = plt.subplots(1, 2, figsize=(17, 4 if len(pdf) < 10 else 6))
    sns.barplot(data=pdf, y=col, x='Cantidad', ax=ejes[0], color='tab:blue')
    ejes[0].set_title(f'Cantidad vs {col}'); ejes[0].grid(axis='x')
    sns.barplot(data=pdf, y=col, x='Pct_>50K', ax=ejes[1], color='tab:orange')
    ejes[1].axvline(df.select(F.avg('Ingreso')).first()[0] * 100, color='k', linestyle='--', label='Promedio general')
    ejes[1].set_title(f'% con ingreso >50K según {col}'); ejes[1].legend(); ejes[1].grid(axis='x')
    plt.tight_layout()
    plt.show()
    return pdf

In [ ]:
tasaPorCategoria(df02, 'Sexo')

In [ ]:
tasaPorCategoria(df02, 'EstadoCivil')

In [ ]:
tasaPorCategoria(df02, 'Parentesco')

In [ ]:
### Educación ordenada por años de estudio
ordenEducacion = [r[0] for r in df02.select('Educacion', 'AniosEducacion').distinct().orderBy('AniosEducacion').collect()]
tasaPorCategoria(df02, 'Educacion', orden=ordenEducacion)

In [ ]:
tasaPorCategoria(df02, 'Ocupacion')

In [ ]:
tasaPorCategoria(df02, 'ClaseTrabajo')

In [ ]:
tasaPorCategoria(df02, 'Raza')

In [ ]:
### País de origen: solo los 15 más frecuentes (los demás son muy pocos)
tasaPorCategoria(df02, 'PaisOrigen').sort_values('Cantidad', ascending=False).head(15)

**Análisis de variables cualitativas**

*(Valores aproximados. Las cifras exactas están en las tablas y gráficas de arriba.)*

- **Sexo:** hay unos 2 hombres por cada mujer. La tasa de `>50K` es casi tres veces mayor en hombres (≈ 30 %)
  que en mujeres (≈ 11 %). Esto refleja la brecha salarial y de participación laboral de 1994. El modelo la
  aprenderá como un patrón de los datos, no como una relación causal, y conviene tenerlo presente por
  temas de **equidad**.
- **EstadoCivil:** es uno de los factores más fuertes. Las personas casadas con cónyuge civil
  (`Married-civ-spouse`) superan los 50K en ≈ 45 % de los casos, frente a ≈ 5 % de quienes nunca se casaron y
  ≈ 10 % de divorciados o separados. Parte del efecto viene de la edad, porque los solteros son más jóvenes, y
  parte de que el ingreso declarado sea del hogar.
- **Parentesco:** repite la información anterior. `Husband` y `Wife` tienen tasas de ≈ 45–47 %, mientras que
  `Own-child` casi nunca pasa de 50K (≈ 1–2 %). Está muy correlacionada con `EstadoCivil` y `Sexo`.
- **Educacion:** la tasa crece de forma casi monótona con el nivel educativo. Es muy baja por debajo de
  `HS-grad`, ≈ 16 % en `HS-grad`, ≈ 41 % en `Bachelors`, ≈ 55 % en `Masters` y ≈ 73–75 % en `Prof-school` y
  `Doctorate`. Esto confirma que `AniosEducacion` es una buena representación ordinal.
- **Ocupacion:** `Exec-managerial` y `Prof-specialty` tienen las tasas más altas (≈ 45–48 %).
  `Priv-house-serv`, `Handlers-cleaners`, `Other-service` y `Farming-fishing` tienen las más bajas (< 12 %).
  Las personas sin ocupación registrada (nulo) tienen una tasa muy baja (≈ 10 %), así que el faltante sí es
  informativo.
- **ClaseTrabajo:** predomina `Private` (≈ 70 %). `Self-emp-inc` (empresarios con sociedad constituida) tiene
  la tasa más alta (≈ 55 %), seguido de `Federal-gov` (≈ 39 %). `Without-pay` y `Never-worked` casi no tienen
  ingresos altos.
- **Raza:** ≈ 85 % de los registros son `White`. `White` y `Asian-Pac-Islander` tienen tasas de ≈ 25–27 %,
  frente a ≈ 12 % en `Black` y `Amer-Indian-Eskimo`. Como en el caso del sexo, es un posible sesgo.
- **PaisOrigen:** ≈ 90 % es `United-States`, con una tasa cercana al promedio. `Mexico` es el segundo país
  (≈ 2 %) y tiene una tasa muy baja (≈ 5 %). Los demás países tienen tan pocos registros que sus tasas no son
  confiables, por eso en la preparación se agrupan.

## 2. Preparación de datos
### 2.1 Eliminación de duplicados

In [ ]:
df03 = df02.dropDuplicates()
print(f"Registros antes: {df02.count()} | después: {df03.count()}")

### 2.2 Imputación de valores faltantes

- `ClaseTrabajo` y `Ocupacion` → nueva categoría **`Desconocido`**. El análisis mostró que no reportar el
  trabajo se asocia a ingresos bajos. Imputar con la moda (`Private` / `Prof-specialty`) borraría esa señal y
  además asignaría la ocupación con una de las tasas más altas.
- `PaisOrigen` → **moda** (`United-States`), porque representa ≈ 90 % de los datos.

In [ ]:
modaPais = df03.where(F.col('PaisOrigen').isNotNull()).groupBy('PaisOrigen').count() \
               .orderBy(F.desc('count')).first()['PaisOrigen']
print("Moda de PaisOrigen:", modaPais)

df04 = df03.fillna({'ClaseTrabajo': 'Desconocido', 'Ocupacion': 'Desconocido', 'PaisOrigen': modaPais})

# Se verifica que no quedan nulos
df04.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df04.columns]).show()

### 2.3 Ingeniería de variables

1. **Se eliminan** `PesoMuestral`, que es un peso del muestreo y no se correlaciona con el ingreso, y
   `Educacion`, que repite la información de `AniosEducacion`.
2. **Transformación logarítmica** de `GananciaCapital` y `PerdidaCapital`: $\log(1+x)$ reduce el sesgo
   extremo y conserva el 0. Se añade la variable binaria `TieneCapital` (1 si hay ganancia o pérdida de
   capital).
3. **Agrupación de PaisOrigen** en `United-States`, `Mexico` y `Otro`. Los otros 39 países suman ≈ 8 % y cada
   uno tiene muy pocos registros. Así se pasa de 41 a 3 categorías y se evitan columnas casi vacías en el
   one-hot.
4. **Agrupación de EstadoCivil**: `Married-civ-spouse` y `Married-AF-spouse` (cónyuge en las fuerzas armadas,
   con muy pocos casos) se unen en `Casado`.

In [ ]:
df05 = df04.drop('PesoMuestral', 'Educacion') \
    .withColumn('LogGanancia', F.log1p(F.col('GananciaCapital'))) \
    .withColumn('LogPerdida',  F.log1p(F.col('PerdidaCapital'))) \
    .withColumn('TieneCapital', ((F.col('GananciaCapital') > 0) | (F.col('PerdidaCapital') > 0)).cast('int')) \
    .withColumn('PaisOrigen', F.when(F.col('PaisOrigen').isin('United-States', 'Mexico'), F.col('PaisOrigen'))
                               .otherwise('Otro')) \
    .withColumn('EstadoCivil', F.when(F.col('EstadoCivil').isin('Married-civ-spouse', 'Married-AF-spouse'), 'Casado')
                                .otherwise(F.col('EstadoCivil'))) \
    .drop('GananciaCapital', 'PerdidaCapital')

### Se deja en memoria el DataFrame limpio: todas las etapas siguientes parten de él
df05 = df05.cache()
print("Registros:", df05.count())
df05.show(5, truncate=False)

In [ ]:
### Verificación de las agrupaciones
df05.groupBy('PaisOrigen').agg(F.count('*').alias('Cantidad'), F.round(F.avg('Ingreso')*100, 2).alias('Pct_>50K')).show()
df05.groupBy('EstadoCivil').agg(F.count('*').alias('Cantidad'), F.round(F.avg('Ingreso')*100, 2).alias('Pct_>50K')).show()
df05.groupBy('TieneCapital').agg(F.count('*').alias('Cantidad'), F.round(F.avg('Ingreso')*100, 2).alias('Pct_>50K')).show()

**Persistencia en HDFS.** El DataFrame limpio se guarda en formato **Parquet** en la carpeta del taller.
Parquet es columnar y comprimido, conserva el esquema y los tipos, y Spark lo lee más rápido que el CSV.
Así las siguientes ejecuciones pueden partir de los datos ya preparados.

In [ ]:
### Se guarda el DataFrame preparado en formato Parquet en HDFS
df05.write.mode("overwrite").parquet("hdfs://10.195.34.34:9000/FormatosHPC/Avila/TallerModelosClasificacion/Parquet")

### Se consulta que quedó en la carpeta
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /FormatosHPC/Avila/TallerModelosClasificacion/Parquet

### 2.4 Codificación de las variables categóricas

- **StringIndexer** convierte cada categoría en un índice numérico (`_IND`). Se usa `handleInvalid='keep'` por
  si aparece una categoría no vista.
- **OneHotEncoder** convierte cada índice en un vector binario (`_OHE`) con `dropLast=True`. Así los modelos
  lineales no interpretan un orden falso entre categorías, por ejemplo entre ocupaciones.

Ambas etapas se encadenan en un **Pipeline** de Spark ML.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

catFinales = ['ClaseTrabajo', 'EstadoCivil', 'Ocupacion', 'Parentesco', 'Raza', 'Sexo', 'PaisOrigen']
numFinales = ['Edad', 'AniosEducacion', 'HorasSemana', 'LogGanancia', 'LogPerdida', 'TieneCapital']

indices = [StringIndexer(inputCol=c, outputCol=c + '_IND', handleInvalid='keep') for c in catFinales]
codificador = OneHotEncoder(inputCols=[c + '_IND' for c in catFinales],
                            outputCols=[c + '_OHE' for c in catFinales], dropLast=True)

### SE GENERA UN PIPELINE
pipelineSalario = Pipeline(stages=indices + [codificador])
df06 = pipelineSalario.fit(df05).transform(df05)
df06.select(numFinales + [c + '_OHE' for c in catFinales] + ['Ingreso']).show(5, truncate=False)

### 2.5 Vector de características

**VectorAssembler** reúne en una sola columna vectorial las 6 variables numéricas y los 7 vectores one-hot. La
etiqueta se convierte a `double`.

In [ ]:
### SE genera un vector con solo las variables independientes
varIndependientes = VectorAssembler(inputCols=numFinales + [c + '_OHE' for c in catFinales],
                                    outputCol='independientes')

vectorIndependientes = varIndependientes.transform(df06) \
                                        .select('independientes', F.col('Ingreso').cast('double').alias('Ingreso'))

### Nombres de cada posición del vector (para interpretar importancias y coeficientes)
metaAttrs = vectorIndependientes.schema['independientes'].metadata['ml_attr']['attrs']
nombresVars = [a['name'] for a in sorted(sum(metaAttrs.values(), []), key=lambda a: a['idx'])]
print(f"Número de variables en el vector: {len(nombresVars)}")
print(nombresVars)

vectorIndependientes.show(5, False)

### 2.6 División de datos de entrenamiento y prueba (estratificada)

Se separa 80 % para entrenamiento y 20 % para prueba **dentro de cada clase**, para conservar la proporción
de `>50K` en ambos conjuntos. Se fija `seed=42` para que sea reproducible.

> Se unieron `adult.data` y `adult.test` y luego se hizo una partición propia. Así la limpieza es una sola y
> la proporción de clases queda controlada. Las dos particiones originales siguen la misma distribución.

In [ ]:
dfML = vectorIndependientes.select(['independientes', 'Ingreso'])

entreno0, prueba0 = dfML.where(F.col('Ingreso') == 0).randomSplit([0.8, 0.2], seed=42)
entreno1, prueba1 = dfML.where(F.col('Ingreso') == 1).randomSplit([0.8, 0.2], seed=42)

entrenamiento = entreno0.union(entreno1)
prueba = prueba0.union(prueba1)

for nombre, conjunto in [('entrenamiento', entrenamiento), ('prueba', prueba)]:
    total = float(conjunto.count())
    altos = conjunto.where('Ingreso == 1').count()
    print(f"Total en {nombre} = {int(total)} | >50K: {altos} ({round(altos/total*100, 2)} %)")

### 2.7 Tratamiento del desbalance de clases

- **Pesos por clase (`weightCol`)**, para RL, DT, RF, GBT, SVC y NB: $w_c = \frac{N}{2\,N_c}$, de modo que
  ambas clases aportan lo mismo a la función de costo. Un registro `>50K` pesa ≈ 3 veces más que uno `<=50K`.
- **Sobremuestreo aleatorio de `>50K`**, para el Perceptrón Multicapa, que no admite `weightCol`.

El balanceo solo se aplica al **entrenamiento**. La prueba conserva la distribución real.

In [ ]:
N  = entrenamiento.count()
N1 = entrenamiento.where(F.col('Ingreso') == 1).count()
N0 = N - N1
peso0 = N / (2.0 * N0)
peso1 = N / (2.0 * N1)
print(f"Peso clase 0 (<=50K) = {round(peso0, 4)}")
print(f"Peso clase 1 (>50K)  = {round(peso1, 4)}")

entrenamiento = entrenamiento.withColumn('peso', F.when(F.col('Ingreso') == 1, F.lit(peso1)).otherwise(F.lit(peso0)))
entrenamiento.groupBy('Ingreso', 'peso').count().show()

### 2.8 Estandarización de las características

**StandardScaler** ($z = \frac{x-\mu}{\sigma}$) se ajusta **solo con el entrenamiento** y se aplica a ambos
conjuntos, para evitar fuga de información. Lo usan RL, SVC, MLP y NB. Los árboles usan las características
originales.

In [ ]:
escalador = StandardScaler(inputCol='independientes', outputCol='independientesEsc', withMean=True, withStd=True)
modeloEscalador = escalador.fit(entrenamiento)

entrenamiento = modeloEscalador.transform(entrenamiento).cache()
prueba        = modeloEscalador.transform(prueba).cache()

### Sobremuestreo de la clase minoritaria (solo para el MLP)
mayoritaria = entrenamiento.where(F.col('Ingreso') == 0)
minoritaria = entrenamiento.where(F.col('Ingreso') == 1)
entrenamientoSobre = mayoritaria.union(
    minoritaria.sample(withReplacement=True, fraction=N0 / N1, seed=42)).cache()

print("Entrenamiento:"); entrenamiento.groupBy('Ingreso').count().orderBy('Ingreso').show()
print("Entrenamiento sobremuestreado (MLP):"); entrenamientoSobre.groupBy('Ingreso').count().orderBy('Ingreso').show()
print("Prueba:"); prueba.groupBy('Ingreso').count().orderBy('Ingreso').show()

## 3. Modelos de MLlib

Se entrenan **10 clasificadores** de `pyspark.ml.classification`, más una línea base sin balanceo:

| # | Modelo | Familia | Balanceo | Características |
|---|---|---|---|---|
| 0 | Regresión Logística (línea base) | Lineal probabilístico | Ninguno | Escaladas |
| 1 | Regresión Logística | Lineal probabilístico | Pesos | Escaladas |
| 2 | Árbol de Decisión | Árboles (CART) | Pesos | Originales |
| 3 | Random Forest | Ensamble *bagging* | Pesos | Originales |
| 4 | Gradient-Boosted Trees | Ensamble *boosting* | Pesos | Originales |
| 5 | Linear SVC | Máximo margen (*hinge loss*) | Pesos | Escaladas |
| 6 | Perceptrón Multicapa | Red neuronal | Sobremuestreo | Escaladas |
| 7 | Naive Bayes Gaussiano | Probabilístico generativo | Pesos | Escaladas |
| 8 | Naive Bayes Multinomial (Laplace) | Probabilístico generativo | Pesos | Originales (≥ 0) |
| 9 | SVC con expansión polinómica grado 2 | Máximo margen no lineal | Pesos | Polinómicas escaladas |
| 10 | Regresión Logística Elastic Net (CV 3-fold) | Lineal regularizado | Pesos | Escaladas |

Además, en la sección 4 se hacen experimentos complementarios con temas del curso: diagnóstico
sesgo–varianza, camino de regularización Lasso, arquitecturas del perceptrón, PCA y agrupamiento (K-Means,
Bisecting K-Means y GMM). Todo se hace con `pyspark.ml`.

### 3.0 Función de evaluación común

Todos los modelos se evalúan con la misma función sobre el mismo conjunto de prueba. La clase positiva es
`>50K`:

- **Accuracy**: proporción total de aciertos. Su referencia es ≈ 0.76, lo que obtendría un modelo que siempre
  responde `<=50K`.
- **Precisión >50K** = TP / (TP + FP): de las personas que el modelo marca como `>50K`, cuántas lo son de verdad.
- **Recall >50K** = TP / (TP + FN): proporción de personas con `>50K` que el modelo detecta.
- **F1 >50K**: media armónica de precisión y *recall*. Es la métrica de equilibrio para la clase positiva.
- **Especificidad** = TN / (TN + FP): proporción de personas `<=50K` que se clasifican bien.
- **AUC-ROC**: capacidad de ordenar a las personas según la probabilidad de `>50K`, sin depender del umbral.
- **AUC-PR**: área bajo la curva precisión–*recall*. Su referencia de azar es la prevalencia (≈ 0.24).

In [ ]:
from pyspark.ml.functions import vector_to_array

resultados = {}      # métricas de cada modelo
puntajes   = {}      # (y_real, score) de cada modelo para las curvas ROC y PR

def evaluarModelo(prediccion, nombre):
    """Calcula las métricas, pinta la matriz de confusión y guarda los resultados del modelo."""
    evalMC = lambda metrica, **kw: MulticlassClassificationEvaluator(
        labelCol='Ingreso', predictionCol='prediction', metricName=metrica, **kw).evaluate(prediccion)

    ### Puntaje continuo: probabilidad de clase 1 si el modelo la entrega; si no (SVC), el margen
    colPuntaje = 'probability' if 'probability' in prediccion.columns else 'rawPrediction'
    evalBin = lambda metrica: BinaryClassificationEvaluator(
        labelCol='Ingreso', rawPredictionCol=colPuntaje, metricName=metrica).evaluate(prediccion)

    accuracy   = evalMC('accuracy')
    precision1 = evalMC('precisionByLabel', metricLabel=1.0)
    recall1    = evalMC('recallByLabel',    metricLabel=1.0)
    f1_1       = evalMC('fMeasureByLabel',  metricLabel=1.0)
    aucROC     = evalBin('areaUnderROC')
    aucPR      = evalBin('areaUnderPR')

    ### Matriz de confusión calculada en Spark
    matriz = np.zeros((2, 2), dtype=int)
    for real, pred, n in prediccion.groupBy('Ingreso', 'prediction').count().collect():
        matriz[int(real), int(pred)] = n
    TN, FP, FN, TP = matriz.ravel()
    especificidad = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    resultados[nombre] = {'Accuracy': accuracy, 'Precision_1': precision1, 'Recall_1': recall1,
                          'F1_1': f1_1, 'Especificidad': especificidad,
                          'AUC_ROC': aucROC, 'AUC_PR': aucPR,
                          'TP': int(TP), 'FN': int(FN), 'FP': int(FP), 'TN': int(TN)}

    ### Puntaje para las curvas ROC y PR
    pdf = prediccion.select('Ingreso', vector_to_array(colPuntaje)[1].alias('score')).toPandas()
    puntajes[nombre] = (pdf['Ingreso'].values, pdf['score'].values)

    print(f"===== {nombre} =====")
    print(f"Accuracy           = {accuracy:.4f}")
    print(f"Precision (>50K)   = {precision1:.4f}")
    print(f"Recall    (>50K)   = {recall1:.4f}")
    print(f"F1        (>50K)   = {f1_1:.4f}")
    print(f"Especificidad      = {especificidad:.4f}")
    print(f"AUC-ROC            = {aucROC:.4f}")
    print(f"AUC-PR             = {aucPR:.4f}")

    plt.figure(figsize=(5, 4))
    sns.heatmap(matriz, annot=True, cbar=False, fmt='d', cmap='Blues',
                xticklabels=['Predicted <=50K', 'Predicted >50K'],
                yticklabels=['True <=50K', 'True >50K'])
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()

### 3.1 Modelo RL (línea base): Regresión Logística sin balanceo

La regresión logística modela $P(y=1\mid x) = \sigma(\theta^T x)$ y ajusta $\theta$ minimizando la entropía
cruzada binaria. Esta versión **no usa pesos**. Sirve para medir cuánto cambia el resultado al balancear.

In [ ]:
from pyspark.ml.classification import LogisticRegression

RLbase = LogisticRegression(featuresCol='independientesEsc', labelCol='Ingreso', maxIter=100)
modeloRLbase = RLbase.fit(entrenamiento)

### Se hacen las predicciones sobre los datos de prueba
prediccionRLbase = modeloRLbase.transform(prueba)
prediccionRLbase.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionRLbase, 'RL base (sin balanceo)')

### 3.2 Modelo RL: Regresión Logística con pesos por clase

Es el mismo modelo con `weightCol='peso'` y una regularización $L_2$ ligera (`regParam=0.01`).

In [ ]:
RL = LogisticRegression(featuresCol='independientesEsc', labelCol='Ingreso', weightCol='peso',
                        maxIter=100, regParam=0.01, elasticNetParam=0.0)
modeloRL = RL.fit(entrenamiento)

prediccionRL = modeloRL.transform(prueba)
prediccionRL.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionRL, 'Regresion Logistica')

### 3.3 Modelo DT: Árbol de Decisión

Construye particiones binarias de forma voraz (tipo CART), eligiendo en cada nodo el corte que más reduce la
impureza de **Gini**. Se limita la profundidad (`maxDepth=8`) y el mínimo por hoja para evitar sobreajuste.

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

DT = DecisionTreeClassifier(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                            maxDepth=8, minInstancesPerNode=20, impurity='gini', seed=42)
modeloDT = DT.fit(entrenamiento)

prediccionDT = modeloDT.transform(prueba)
prediccionDT.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionDT, 'Arbol de Decision')

### 3.4 Modelo RF: Random Forest

Ensamble *bagging*: cada árbol se entrena con una muestra *bootstrap* y un subconjunto aleatorio de variables
por partición (`featureSubsetStrategy='sqrt'`). Así se reduce la varianza al promediar árboles poco
correlacionados. Spark entrena los árboles de forma distribuida.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

RF = RandomForestClassifier(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                            numTrees=100, maxDepth=10, minInstancesPerNode=5,
                            featureSubsetStrategy='sqrt', seed=42)
modeloRF = RF.fit(entrenamiento)

prediccionRF = modeloRF.transform(prueba)
prediccionRF.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionRF, 'Random Forest')

### 3.5 Modelo GBT: Gradient-Boosted Trees

Ensamble *boosting*: añade árboles poco profundos uno tras otro, cada uno ajustado al gradiente negativo de la
pérdida logística de los anteriores. Así reduce el sesgo de forma secuencial. Se usan 100 iteraciones,
profundidad 5 y tasa de aprendizaje 0.1.

In [ ]:
from pyspark.ml.classification import GBTClassifier

GBT = GBTClassifier(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                    maxIter=100, maxDepth=5, stepSize=0.1, subsamplingRate=0.8, seed=42)
modeloGBT = GBT.fit(entrenamiento)

prediccionGBT = modeloGBT.transform(prueba)
prediccionGBT.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionGBT, 'GBT')

### 3.6 Modelo SVC: Máquina de Soporte Vectorial lineal

`LinearSVC` busca el hiperplano de **máximo margen** con margen suave. Minimiza la *hinge loss* más una
penalización $L_2$ (`regParam`). MLlib solo ofrece kernel lineal y no entrega probabilidades, así que para
ROC y PR se usa el margen (`rawPrediction`).

In [ ]:
from pyspark.ml.classification import LinearSVC

SVC = LinearSVC(featuresCol='independientesEsc', labelCol='Ingreso', weightCol='peso',
                maxIter=100, regParam=0.01)
modeloSVC = SVC.fit(entrenamiento)

prediccionSVC = modeloSVC.transform(prueba)
prediccionSVC.select('Ingreso', 'rawPrediction', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionSVC, 'Linear SVC')

### 3.7 Modelo MPC: Perceptrón Multicapa

Red neuronal *feed-forward* con activación sigmoide en las capas ocultas y *softmax* en la salida, entrenada
con L-BFGS. La capa de entrada tiene tantas neuronas como variables hay en el vector, luego vienen capas
ocultas de 16 y 8 neuronas y una salida de 2 clases. Se entrena con el conjunto **sobremuestreado**.

In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

capas = [len(nombresVars), 16, 8, 2]
print(f"Arquitectura del MLP: {capas}")

MPC = MultilayerPerceptronClassifier(featuresCol='independientesEsc', labelCol='Ingreso',
                                     layers=capas, maxIter=200, blockSize=128, seed=42)
modeloMPC = MPC.fit(entrenamientoSobre)

prediccionMPC = modeloMPC.transform(prueba)
prediccionMPC.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionMPC, 'Perceptron Multicapa')

### 3.8 Modelo NB: Naive Bayes Gaussiano

Clasificador generativo que aplica el teorema de Bayes suponiendo que las variables son independientes dada
la clase, cada una con distribución normal. Es muy rápido y sirve como referencia simple.

In [ ]:
from pyspark.ml.classification import NaiveBayes

NB = NaiveBayes(featuresCol='independientesEsc', labelCol='Ingreso', weightCol='peso', modelType='gaussian')
modeloNB = NB.fit(entrenamiento)

prediccionNB = modeloNB.transform(prueba)
prediccionNB.select('Ingreso', 'probability', 'prediction').show(5, False)

In [ ]:
evaluarModelo(prediccionNB, 'Naive Bayes')

### 3.9 Modelo NBM: Naive Bayes Multinomial con suavizado de Laplace

Naive Bayes es un modelo **generativo**: no modela $P(y\mid x)$ directamente, como la Regresión Logística
(modelo **discriminativo**), sino la distribución conjunta $P(x,y) = P(x\mid y)\,P(y)$. Luego aplica la regla
de Bayes:

$$P(y\mid x) = \frac{P(y)\prod_j P(x_j\mid y)}{P(x)}$$

- $P(y)$ es la probabilidad **a priori** de cada clase, $\prod_j P(x_j\mid y)$ es la **verosimilitud**
  factorizada (supuesto de **independencia condicional**) y $P(y\mid x)$ es la **posterior**.
- La variante **multinomial** trata cada variable como un conteo no negativo, así que se usan las
  características **sin estandarizar**, que son todas $\geq 0$.
- El **suavizado de Laplace** (`smoothing`) suma $\alpha$ a cada conteo para que ninguna probabilidad
  condicional sea 0. Sin él, una sola categoría no observada en una clase anularía todo el producto.

Se comparan varios valores de $\alpha$ y se toma el mejor según AUC-ROC.

In [ ]:
from pyspark.ml.classification import NaiveBayes

aucEval = BinaryClassificationEvaluator(labelCol='Ingreso', rawPredictionCol='probability', metricName='areaUnderROC')

filasNB = []
for alfa in [0.0, 0.5, 1.0, 5.0, 20.0]:
    m = NaiveBayes(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                   modelType='multinomial', smoothing=alfa).fit(entrenamiento)
    filasNB.append({'smoothing (alfa)': alfa, 'AUC_ROC prueba': aucEval.evaluate(m.transform(prueba))})
tablaNB = pd.DataFrame(filasNB)
print(tablaNB.round(4).to_string(index=False))

alfaOpt = tablaNB.loc[tablaNB['AUC_ROC prueba'].idxmax(), 'smoothing (alfa)']
print(f"\nMejor suavizado: alfa = {alfaOpt}")

modeloNBM = NaiveBayes(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                       modelType='multinomial', smoothing=float(alfaOpt)).fit(entrenamiento)
prediccionNBM = modeloNBM.transform(prueba)

In [ ]:
evaluarModelo(prediccionNBM, 'Naive Bayes Multinomial')

### 3.10 Modelo SVC-Poly: SVM con expansión polinómica explícita (kernel polinomial)

En la **formulación dual** de la SVM, el entrenamiento y la predicción dependen solo de productos internos
$\langle x^{[i]}, x^{[j]}\rangle$. El **truco del kernel** cambia ese producto por
$K(x,z) = \langle\phi(x),\phi(z)\rangle$, lo que equivale a trabajar en un espacio de mayor dimensión sin
calcular $\phi(x)$. Mercer garantiza que es válido si $K$ es semidefinido positivo.

`LinearSVC` de MLlib resuelve el problema **primal** con *hinge loss* y no admite kernels. Se resuelve el
problema de **escala**: un kernel exige la matriz de Gram de $n \times n$, que para decenas de miles de
registros no es viable en un entorno distribuido. La alternativa en Spark es construir **explícitamente**
$\phi(x)$ con `PolynomialExpansion` de grado 2, que agrega todos los cuadrados y los productos cruzados
$x_i x_j$. Es el mismo espacio de características que el kernel polinomial
$K(x,z) = (\langle x,z\rangle + c)^2$, solo que calculado de forma directa. Así la frontera de decisión
deja de ser lineal en el espacio original.

Se arma un **Pipeline**: expansión → escalado (`withMean=False` para mantener el vector disperso) → LinearSVC.

In [ ]:
from pyspark.ml.feature import PolynomialExpansion
from pyspark.ml.classification import LinearSVC

expansion   = PolynomialExpansion(degree=2, inputCol='independientes', outputCol='polinomio')
escaladoPol = StandardScaler(inputCol='polinomio', outputCol='polinomioEsc', withMean=False, withStd=True)
svcPol      = LinearSVC(featuresCol='polinomioEsc', labelCol='Ingreso', weightCol='peso', maxIter=100, regParam=0.01)

modeloSVCpoly = Pipeline(stages=[expansion, escaladoPol, svcPol]).fit(entrenamiento)
print("Dimensión original  :", len(nombresVars))
print("Dimensión expandida :", modeloSVCpoly.stages[-1].numFeatures)

prediccionSVCpoly = modeloSVCpoly.transform(prueba)

In [ ]:
evaluarModelo(prediccionSVCpoly, 'SVC polinomial (grado 2)')

### 3.11 Ajuste de hiperparámetros con validación cruzada *k-fold*: Ridge, Lasso y Elastic Net

**Diseño experimental.** Si los hiperparámetros se eligen mirando el conjunto de prueba, este deja de ser una
estimación honesta del error de generalización. Por eso se usa **validación cruzada de k = 3 pliegues** solo
sobre el entrenamiento: se divide en 3 partes, se entrena con 2 y se valida con la tercera, rotando. Se elige
la combinación con mejor AUC promedio y se reentrena con todo el entrenamiento. El conjunto de prueba se usa
una sola vez, al final.

La Regresión Logística de MLlib minimiza la entropía cruzada más una penalización **Elastic Net**:

$$J(\theta) = -\frac{1}{m}\sum_i \big[y_i\log h_\theta(x_i) + (1-y_i)\log(1-h_\theta(x_i))\big]
+ \lambda\Big[\alpha\,\lVert\theta\rVert_1 + \frac{1-\alpha}{2}\lVert\theta\rVert_2^2\Big]$$

- `regParam` $= \lambda$ es la fuerza de la regularización. Más $\lambda$ significa más **sesgo** y menos
  **varianza**.
- `elasticNetParam` $= \alpha$: con $\alpha = 0$ es **Ridge ($L_2$)**, que contrae los pesos sin anularlos;
  con $\alpha = 1$ es **Lasso ($L_1$)**, que lleva pesos a 0 y funciona como **selector de variables**;
  los valores intermedios combinan ambas.

`ParamGridBuilder` arma la malla y `CrossValidator` la recorre de forma distribuida (`parallelism`).

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.classification import LogisticRegression

rlCV = LogisticRegression(featuresCol='independientesEsc', labelCol='Ingreso', weightCol='peso', maxIter=100)

malla = ParamGridBuilder() \
    .addGrid(rlCV.regParam, [0.0001, 0.001, 0.01, 0.1]) \
    .addGrid(rlCV.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

validadorCruzado = CrossValidator(estimator=rlCV, estimatorParamMaps=malla,
                                  evaluator=BinaryClassificationEvaluator(labelCol='Ingreso', metricName='areaUnderROC'),
                                  numFolds=3, parallelism=4, seed=42)
modeloCV = validadorCruzado.fit(entrenamiento)

### Resultados de la malla
tablaCV = pd.DataFrame([{'regParam (lambda)': p[rlCV.regParam], 'elasticNetParam (alfa)': p[rlCV.elasticNetParam],
                         'AUC promedio CV': m} for p, m in zip(malla, modeloCV.avgMetrics)])
tablaPivote = tablaCV.pivot(index='regParam (lambda)', columns='elasticNetParam (alfa)', values='AUC promedio CV')

plt.figure(figsize=(7, 4))
sns.heatmap(tablaPivote, annot=True, fmt='.4f', cmap='viridis')
plt.title('AUC promedio en validación cruzada (3-fold)')
plt.xlabel('elasticNetParam (0 = Ridge, 1 = Lasso)')
plt.show()

mejorRL = modeloCV.bestModel
print(f"Mejores hiperparámetros: regParam = {mejorRL.getRegParam()}, elasticNetParam = {mejorRL.getElasticNetParam()}")
prediccionRLcv = mejorRL.transform(prueba)

In [ ]:
evaluarModelo(prediccionRLcv, 'RL Elastic Net (CV)')

## 4. Evaluación de modelos
### 4.1 Tabla comparativa de métricas (conjunto de prueba)

In [ ]:
tablaResultados = pd.DataFrame(resultados).T
tablaResultados = tablaResultados[['Accuracy', 'Precision_1', 'Recall_1', 'F1_1', 'Especificidad',
                                   'AUC_ROC', 'AUC_PR', 'TP', 'FN', 'FP', 'TN']]
tablaResultados[['TP', 'FN', 'FP', 'TN']] = tablaResultados[['TP', 'FN', 'FP', 'TN']].astype(int)
tablaResultados = tablaResultados.sort_values('AUC_ROC', ascending=False)
tablaResultados.round(4)

### 4.2 Comparación gráfica de las métricas principales

In [ ]:
metricasGraf = ['Accuracy', 'Recall_1', 'Precision_1', 'F1_1', 'AUC_ROC', 'AUC_PR']
ax = tablaResultados[metricasGraf].plot(kind='bar', figsize=(15, 5), width=0.8)
plt.title('Métricas por modelo (conjunto de prueba)')
plt.ylabel('Valor')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.05)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=len(metricasGraf))
plt.grid(axis='y')
plt.tight_layout()
plt.show()

### 4.3 Curvas ROC y Precisión–Recall

Las curvas muestran el comportamiento de cada modelo **en todos los umbrales**. La línea punteada es el
clasificador aleatorio.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, ejes = plt.subplots(1, 2, figsize=(16, 6))
for nombre, (yReal, score) in puntajes.items():
    fpr, tpr, _ = roc_curve(yReal, score)
    ejes[0].plot(fpr, tpr, label=f"{nombre} (AUC={auc(fpr, tpr):.3f})")
    prec, rec, _ = precision_recall_curve(yReal, score)
    ejes[1].plot(rec, prec, label=f"{nombre} (AUC={auc(rec, prec):.3f})")

ejes[0].plot([0, 1], [0, 1], 'k--', label='Azar')
ejes[0].set_title('Curva ROC'); ejes[0].set_xlabel('Tasa de falsos positivos (1 - Especificidad)')
ejes[0].set_ylabel('Tasa de verdaderos positivos (Recall)'); ejes[0].legend(fontsize=9); ejes[0].grid()

prevalencia = np.mean(next(iter(puntajes.values()))[0])
ejes[1].axhline(prevalencia, color='k', linestyle='--', label=f'Azar (prevalencia={prevalencia:.3f})')
ejes[1].set_title('Curva Precisión-Recall'); ejes[1].set_xlabel('Recall')
ejes[1].set_ylabel('Precisión'); ejes[1].legend(fontsize=9); ejes[1].grid()
plt.tight_layout()
plt.show()

### 4.4 Interpretabilidad: importancia de variables y coeficientes

- **RF / GBT**: la importancia de cada variable es la reducción total de impureza que logra, normalizada.
- **Regresión Logística**: los coeficientes se calcularon sobre variables estandarizadas, así que se pueden
  comparar. Un coeficiente positivo aumenta la probabilidad de `>50K` y uno negativo la disminuye.

In [ ]:
importancias = pd.DataFrame({
    'Random Forest': modeloRF.featureImportances.toArray(),
    'GBT':           modeloGBT.featureImportances.toArray(),
}, index=nombresVars)
coeficientesRL = pd.Series(modeloRL.coefficients.toArray(), index=nombresVars)

top = importancias['GBT'].sort_values(ascending=False).head(20).index
topCoef = coeficientesRL.abs().sort_values(ascending=False).head(20).index

fig, ejes = plt.subplots(1, 2, figsize=(18, 8))
importancias.loc[top].sort_values('GBT').plot(kind='barh', ax=ejes[0])
ejes[0].set_title('Top 20 importancia de variables (RF y GBT)'); ejes[0].grid(axis='x')

coefOrden = coeficientesRL.loc[topCoef].sort_values()
coefOrden.plot(kind='barh', ax=ejes[1], color=['tab:red' if v < 0 else 'tab:blue' for v in coefOrden])
ejes[1].set_title('Top 20 coeficientes de la Regresión Logística (estandarizados)'); ejes[1].grid(axis='x')
plt.tight_layout()
plt.show()

importancias.sort_values('GBT', ascending=False).head(15).round(4)

### 4.5 Ajuste del umbral de decisión (mejor modelo por AUC-ROC)

El umbral de 0.5 no siempre es el mejor. Se recorren varios umbrales del mejor modelo para ver cómo se
intercambian la precisión y el *recall*, y cuál maximiza el F1.

In [ ]:
mejorModelo = tablaResultados.drop('RL base (sin balanceo)', errors='ignore')['AUC_ROC'].idxmax()
yReal, score = puntajes[mejorModelo]
print(f"Modelo analizado: {mejorModelo}")

umbrales = np.round(np.arange(0.1, 0.95, 0.05), 2) if score.min() >= 0 and score.max() <= 1 \
           else np.quantile(score, np.arange(0.1, 0.95, 0.05))

filas = []
for u in umbrales:
    yPred = (score >= u).astype(int)
    TP = int(((yPred == 1) & (yReal == 1)).sum()); FN = int(((yPred == 0) & (yReal == 1)).sum())
    FP = int(((yPred == 1) & (yReal == 0)).sum()); TN = int(((yPred == 0) & (yReal == 0)).sum())
    rec = TP / (TP + FN) if TP + FN else 0; pre = TP / (TP + FP) if TP + FP else 0
    filas.append({'Umbral': round(float(u), 3), 'Accuracy': (TP + TN) / len(yReal),
                  'Recall_1': rec, 'Precision_1': pre, 'F1_1': 2*pre*rec/(pre+rec) if pre + rec else 0,
                  'Especificidad': TN / (TN + FP)})
tablaUmbral = pd.DataFrame(filas)
print("Umbral con mayor F1:", tablaUmbral.loc[tablaUmbral['F1_1'].idxmax(), 'Umbral'])
tablaUmbral.round(4)

### 4.6 Diagnóstico sesgo–varianza

El error de un modelo se puede ver como **sesgo** (qué tan rígido es) más **varianza** (qué tan sensible es a
la muestra de entrenamiento):

| Situación | Entrenamiento | Prueba | Diagnóstico |
|---|---|---|---|
| Sesgo alto (*underfitting*) | desempeño bajo | desempeño bajo | modelo demasiado simple |
| Varianza alta (*overfitting*) | desempeño muy alto | bastante peor | memoriza ruido |
| Buen ajuste | alto | parecido al de entrenamiento | generaliza |

Primero se compara el AUC de cada modelo en entrenamiento y en prueba. Una **brecha** grande indica
sobreajuste.

In [ ]:
modelosEntrenados = {
    'Regresion Logistica': modeloRL, 'RL Elastic Net (CV)': mejorRL, 'Arbol de Decision': modeloDT,
    'Random Forest': modeloRF, 'GBT': modeloGBT, 'Linear SVC': modeloSVC, 'SVC polinomial (grado 2)': modeloSVCpoly,
    'Perceptron Multicapa': modeloMPC, 'Naive Bayes': modeloNB, 'Naive Bayes Multinomial': modeloNBM}

def aucDe(modelo, datos):
    pred = modelo.transform(datos)
    col = 'probability' if 'probability' in pred.columns else 'rawPrediction'
    return BinaryClassificationEvaluator(labelCol='Ingreso', rawPredictionCol=col, metricName='areaUnderROC').evaluate(pred)

filasSV = []
for nombre, modelo in modelosEntrenados.items():
    aTrain, aTest = aucDe(modelo, entrenamiento), aucDe(modelo, prueba)
    filasSV.append({'Modelo': nombre, 'AUC entrenamiento': aTrain, 'AUC prueba': aTest, 'Brecha': aTrain - aTest})
tablaSV = pd.DataFrame(filasSV).set_index('Modelo').sort_values('Brecha', ascending=False)

tablaSV[['AUC entrenamiento', 'AUC prueba']].plot(kind='barh', figsize=(10, 6))
plt.title('AUC en entrenamiento vs prueba (brecha = varianza)'); plt.xlim(0.5, 1.0); plt.grid(axis='x')
plt.show()
tablaSV.round(4)

**Curva de complejidad del Árbol de Decisión.** La profundidad máxima controla la complejidad del árbol. Con
poca profundidad hay sesgo alto, y con mucha el árbol memoriza el entrenamiento (varianza alta). La
profundidad adecuada es la que maximiza el desempeño en validación, antes de que la curva de prueba empiece a
caer mientras la de entrenamiento sigue subiendo.

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

### Se separa un conjunto de VALIDACIÓN dentro del entrenamiento para no usar la prueba en la elección
subEntreno, validacion = entrenamiento.randomSplit([0.8, 0.2], seed=7)

profundidades = list(range(1, 16))
curvaTrain, curvaVal = [], []
for d in profundidades:
    m = DecisionTreeClassifier(featuresCol='independientes', labelCol='Ingreso', weightCol='peso',
                               maxDepth=d, seed=42).fit(subEntreno)
    curvaTrain.append(aucDe(m, subEntreno)); curvaVal.append(aucDe(m, validacion))

dOpt = profundidades[int(np.argmax(curvaVal))]
plt.figure(figsize=(10, 4))
plt.plot(profundidades, curvaTrain, 'o-', label='Entrenamiento')
plt.plot(profundidades, curvaVal, 's-', label='Validación')
plt.axvline(dOpt, color='k', linestyle='--', label=f'Mejor profundidad = {dOpt}')
plt.title('Curva de complejidad: Árbol de Decisión'); plt.xlabel('maxDepth'); plt.ylabel('AUC-ROC')
plt.legend(); plt.grid(); plt.show()

**Curva de aprendizaje del boosting (GBT).** En el *gradient boosting* cada iteración añade un árbol ajustado a
los **pseudo-residuos**, es decir, al gradiente negativo de la pérdida. `evaluateEachIteration` devuelve la
pérdida logística después de cada árbol. En entrenamiento baja siempre. En prueba baja hasta un punto y luego
se estanca o sube. Ese punto es la cantidad de iteraciones adecuada y equivale a una **parada temprana**
(*early stopping*).

In [ ]:
perdidaTrain = modeloGBT.evaluateEachIteration(entrenamiento)
perdidaTest  = modeloGBT.evaluateEachIteration(prueba)
itOpt = int(np.argmin(perdidaTest)) + 1

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(perdidaTrain) + 1), perdidaTrain, label='Entrenamiento')
plt.plot(range(1, len(perdidaTest) + 1), perdidaTest, label='Prueba')
plt.axvline(itOpt, color='k', linestyle='--', label=f'Mínimo en prueba: iteración {itOpt}')
plt.title('GBT: pérdida logística por iteración'); plt.xlabel('Número de árboles'); plt.ylabel('Log-loss')
plt.legend(); plt.grid(); plt.show()

### 4.7 Regularización $L_1$ (Lasso) como selector de variables

Se entrena la Regresión Logística con Lasso puro (`elasticNetParam=1`) para valores crecientes de
$\lambda$. A medida que $\lambda$ aumenta, cada vez más coeficientes quedan **exactamente en 0** (*sparsity*):
el modelo descarta variables. Las últimas en desaparecer son las más importantes. Ridge, en cambio, reduce los
coeficientes pero casi nunca los deja en 0.

In [ ]:
lambdas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
caminoL1, activas, aucL1 = [], [], []
for lam in lambdas:
    m = LogisticRegression(featuresCol='independientesEsc', labelCol='Ingreso', weightCol='peso',
                           maxIter=100, regParam=lam, elasticNetParam=1.0).fit(entrenamiento)
    coefs = m.coefficients.toArray()
    caminoL1.append(coefs); activas.append(int((np.abs(coefs) > 1e-8).sum())); aucL1.append(aucDe(m, prueba))
caminoL1 = pd.DataFrame(caminoL1, index=lambdas, columns=nombresVars)

fig, ejes = plt.subplots(1, 2, figsize=(17, 5))
for c in caminoL1.columns:
    ejes[0].plot(lambdas, caminoL1[c], label=c if caminoL1[c].abs().max() > caminoL1.abs().values.max() * 0.25 else None)
ejes[0].set_xscale('log'); ejes[0].set_title('Camino de coeficientes Lasso'); ejes[0].set_xlabel('lambda (regParam)')
ejes[0].legend(fontsize=8); ejes[0].grid()
ejes[1].plot(lambdas, activas, 'o-', color='tab:green'); ejes[1].set_xscale('log')
ejes[1].set_title('Variables activas (coeficiente distinto de 0)'); ejes[1].set_xlabel('lambda'); ejes[1].grid()
eje2 = ejes[1].twinx(); eje2.plot(lambdas, aucL1, 's--', color='tab:red'); eje2.set_ylabel('AUC prueba', color='tab:red')
plt.tight_layout(); plt.show()

print("Variables que sobreviven con el lambda más alto:")
print(caminoL1.iloc[-1][caminoL1.iloc[-1].abs() > 1e-8].sort_values(key=abs, ascending=False).round(4).to_string())

### 4.8 Del perceptrón al perceptrón multicapa: efecto de la arquitectura y del optimizador

- El **perceptrón de Rosenblatt** y **Adaline** son neuronas de una sola capa, así que solo pueden trazar
  fronteras **lineales**. Por eso no resuelven el problema del **XOR**. En MLlib, `layers=[n, 2]` (sin capas
  ocultas, con *softmax*) es justamente un modelo lineal, equivalente a una regresión logística.
- Al añadir **capas ocultas** con activación no lineal (sigmoide en MLlib), el MLP puede representar
  fronteras no lineales. Los pesos se ajustan con **retropropagación** (regla de la cadena).
- `solver='l-bfgs'` es un método cuasi-Newton de segundo orden. `solver='gd'` es el **descenso de gradiente**
  clásico con tasa de aprendizaje `stepSize`, que es la regla Delta de Adaline generalizada a varias capas.
- `maxIter` limitado actúa como **parada temprana**, que es una forma de regularización.

In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

nEnt = len(nombresVars)
configuraciones = [
    ('Perceptron sin capa oculta [n,2]',  [nEnt, 2],         'l-bfgs', 200),
    ('1 capa oculta [n,8,2]',             [nEnt, 8, 2],      'l-bfgs', 200),
    ('2 capas ocultas [n,16,8,2]',        [nEnt, 16, 8, 2],  'l-bfgs', 200),
    ('3 capas ocultas [n,32,16,8,2]',     [nEnt, 32, 16, 8, 2], 'l-bfgs', 200),
    ('2 capas ocultas, descenso gradiente', [nEnt, 16, 8, 2], 'gd',    200),
    ('2 capas ocultas, parada temprana (20 it)', [nEnt, 16, 8, 2], 'l-bfgs', 20),
]
filasMLP = []
for nombre, capasCfg, solver, it in configuraciones:
    m = MultilayerPerceptronClassifier(featuresCol='independientesEsc', labelCol='Ingreso', layers=capasCfg,
                                       solver=solver, maxIter=it, stepSize=0.03, blockSize=128, seed=42).fit(entrenamientoSobre)
    filasMLP.append({'Configuración': nombre, 'Parámetros': sum(a*b + b for a, b in zip(capasCfg[:-1], capasCfg[1:])),
                     'AUC entrenamiento': aucDe(m, entrenamiento), 'AUC prueba': aucDe(m, prueba)})
tablaMLP = pd.DataFrame(filasMLP).set_index('Configuración')
tablaMLP.round(4)

### 4.9 Reducción de dimensionalidad con PCA

Con muchas variables (sobre todo después del one-hot) aparece la **maldición de la dimensionalidad**: los
datos quedan dispersos y las distancias pierden contraste. El **Análisis de Componentes Principales** busca
las direcciones de máxima varianza resolviendo $C\,u = \lambda\,u$ sobre la matriz de covarianza $C$:

- los **eigenvectores** $u$ son las direcciones (cargas) de los componentes;
- los **eigenvalores** $\lambda$ son la varianza que explica cada uno;
- la proyección es $z = X\cdot u$.

PCA exige **centrar y estandarizar** antes ($Z$-score), por eso se aplica sobre `independientesEsc`. PCA es
**no supervisado**: maximiza la varianza sin mirar la etiqueta, así que los primeros componentes no
necesariamente son los que mejor separan las clases. Para comprobarlo se entrena una regresión logística con
distintos números de componentes.

In [ ]:
from pyspark.ml.feature import PCA

pcaCompleto = PCA(k=nEnt, inputCol='independientesEsc', outputCol='pca').fit(entrenamiento)
varExp = pcaCompleto.explainedVariance.toArray()
varAcum = np.cumsum(varExp)
k90 = int(np.searchsorted(varAcum, 0.90) + 1)

plt.figure(figsize=(12, 4))
plt.bar(range(1, nEnt + 1), varExp, label='Varianza explicada')
plt.step(range(1, nEnt + 1), varAcum, where='mid', color='tab:red', label='Varianza acumulada')
plt.axhline(0.90, color='k', linestyle='--'); plt.axvline(k90, color='k', linestyle=':', label=f'90 % con k = {k90}')
plt.title('PCA: varianza explicada por componente'); plt.xlabel('Componente'); plt.legend(); plt.grid(); plt.show()

### Cargas de los dos primeros componentes (las variables que más pesan en cada uno)
cargas = pd.DataFrame(pcaCompleto.pc.toArray()[:, :2], index=nombresVars, columns=['PC1', 'PC2'])
print(cargas.reindex(cargas['PC1'].abs().sort_values(ascending=False).index).head(8).round(3))

In [ ]:
### Proyección en 2D (muestra) coloreada por la etiqueta
pca2 = PCA(k=2, inputCol='independientesEsc', outputCol='pca2').fit(entrenamiento)
muestra = pca2.transform(prueba).select(vector_to_array('pca2').alias('z'), 'Ingreso').toPandas()
muestra['PC1'] = muestra['z'].apply(lambda z: z[0]); muestra['PC2'] = muestra['z'].apply(lambda z: z[1])

plt.figure(figsize=(8, 6))
sns.scatterplot(data=muestra.sort_values('Ingreso'), x='PC1', y='PC2', hue='Ingreso', alpha=0.5, s=15,
                palette={0.0: 'tab:blue', 1.0: 'tab:red'})
plt.title('Proyección PCA en 2 componentes (conjunto de prueba)'); plt.grid(); plt.show()

In [ ]:
### Desempeño de una RL entrenada sobre k componentes principales
filasPCA = []
for k in sorted(set([1, 2, 3, 5, 10, k90, nEnt])):
    if k > nEnt: continue
    pipe = Pipeline(stages=[PCA(k=k, inputCol='independientesEsc', outputCol='pcaK'),
                            LogisticRegression(featuresCol='pcaK', labelCol='Ingreso', weightCol='peso', maxIter=100)])
    m = pipe.fit(entrenamiento)
    filasPCA.append({'k componentes': k, 'Varianza acumulada': varAcum[k - 1], 'AUC prueba': aucDe(m, prueba)})
tablaPCA = pd.DataFrame(filasPCA)
tablaPCA.round(4)

### 4.10 Agrupamiento no supervisado: K-Means, Bisecting K-Means y Mezclas Gaussianas

Se busca si existen **grupos naturales** en los datos sin usar la etiqueta, para luego ver qué proporción de
`>50K` hay en cada grupo. Esto sirve como segmentación.

- **K-Means**: asigna cada punto al centroide euclidiano más cercano y recalcula cada centroide como la media.
  Así minimiza la suma de errores cuadráticos intra-clúster (SSE). MLlib usa **K-Means||**, la versión
  distribuida de K-Means++, para inicializar.
- **Elección de k**: el **método del codo** busca dónde la SSE deja de bajar de forma marcada. El
  **coeficiente de silueta** va de -1 a 1 y es mayor cuando los clústeres son compactos y están bien
  separados.
- **Bisecting K-Means**: agrupamiento **jerárquico divisivo**. Empieza con un solo clúster y lo divide en dos
  de forma recursiva, que es la versión de arriba hacia abajo del dendrograma.
- **Gaussian Mixture (GMM)**: agrupamiento **probabilístico suave**. Estima pesos, medias y covarianzas con el
  algoritmo **EM** (Esperanza–Maximización) y cada punto recibe una probabilidad de pertenencia a cada
  componente. Como la covarianza completa en muchas dimensiones es costosa e inestable, se ajusta sobre los
  primeros componentes de PCA.
- DBSCAN y HDBSCAN no están en MLlib. Necesitan calcular vecindarios entre todos los puntos, algo poco
  escalable de forma distribuida.

In [ ]:
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator

evalSilueta = ClusteringEvaluator(featuresCol='independientesEsc', predictionCol='cluster', metricName='silhouette')

filasKM = []
for k in range(2, 8 + 1):
    km = KMeans(featuresCol='independientesEsc', predictionCol='cluster', k=k, seed=42, maxIter=50).fit(entrenamiento)
    filasKM.append({'k': k, 'SSE': km.summary.trainingCost, 'Silueta': evalSilueta.evaluate(km.transform(entrenamiento))})
tablaKM = pd.DataFrame(filasKM)
kOpt = int(tablaKM.loc[tablaKM['Silueta'].idxmax(), 'k'])

fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
ejes[0].plot(tablaKM['k'], tablaKM['SSE'], 'o-'); ejes[0].set_title('Método del codo (SSE)'); ejes[0].set_xlabel('k'); ejes[0].grid()
ejes[1].plot(tablaKM['k'], tablaKM['Silueta'], 's-', color='tab:orange'); ejes[1].axvline(kOpt, color='k', linestyle='--')
ejes[1].set_title(f'Coeficiente de silueta (mejor k = {kOpt})'); ejes[1].set_xlabel('k'); ejes[1].grid()
plt.tight_layout(); plt.show()
tablaKM.round(4)

In [ ]:
### Se ajustan los tres algoritmos con el k elegido
kmFinal  = KMeans(featuresCol='independientesEsc', predictionCol='cluster', k=kOpt, seed=42, maxIter=50).fit(entrenamiento)
bkmFinal = BisectingKMeans(featuresCol='independientesEsc', predictionCol='cluster', k=kOpt, seed=42).fit(entrenamiento)

pcaGMM = PCA(k=min(5, nEnt), inputCol='independientesEsc', outputCol='pcaGMM').fit(entrenamiento)
datosGMM = pcaGMM.transform(entrenamiento)
gmmFinal = GaussianMixture(featuresCol='pcaGMM', predictionCol='cluster', probabilityCol='probCluster',
                           k=kOpt, seed=42, maxIter=100).fit(datosGMM)

def perfilClusters(pred, nombre):
    """Tamaño, % de la clase positiva y medias de las primeras variables numéricas por clúster."""
    arr = pred.withColumn('x', vector_to_array('independientes'))
    aggs = [F.count('*').alias('Tamaño'), F.round(F.avg('Ingreso') * 100, 2).alias('% >50K')] + \
           [F.round(F.avg(F.col('x')[i]), 2).alias(nombresVars[i]) for i in range(min(5, nEnt))]
    return arr.groupBy('cluster').agg(*aggs).orderBy('cluster').toPandas().assign(Algoritmo=nombre)

perfiles = pd.concat([
    perfilClusters(kmFinal.transform(entrenamiento), 'K-Means'),
    perfilClusters(bkmFinal.transform(entrenamiento), 'Bisecting K-Means'),
    perfilClusters(gmmFinal.transform(datosGMM), 'GMM (sobre PCA)'),
])
print("Silueta K-Means          :", round(evalSilueta.evaluate(kmFinal.transform(entrenamiento)), 4))
print("Silueta Bisecting K-Means:", round(evalSilueta.evaluate(bkmFinal.transform(entrenamiento)), 4))
print("Pesos de la mezcla GMM   :", np.round(gmmFinal.weights, 3))
perfiles.set_index(['Algoritmo', 'cluster'])

In [ ]:
### % de la clase positiva por clúster en cada algoritmo
plt.figure(figsize=(12, 4))
sns.barplot(data=perfiles, x='cluster', y='% >50K', hue='Algoritmo')
plt.axhline(entrenamiento.select(F.avg('Ingreso')).first()[0] * 100, color='k', linestyle='--', label='Promedio general')
plt.title('Proporción de la clase positiva por clúster'); plt.legend(); plt.grid(axis='y'); plt.show()

### Incertidumbre del GMM: probabilidad máxima de pertenencia (soft clustering)
probMax = gmmFinal.transform(datosGMM).select(F.array_max(vector_to_array('probCluster')).alias('pmax')).toPandas()
plt.figure(figsize=(8, 3))
sns.histplot(probMax['pmax'], bins=30)
plt.title('GMM: probabilidad máxima de pertenencia por registro'); plt.xlabel('max P(cluster | x)'); plt.grid(); plt.show()

**Lectura de los experimentos complementarios**
- **Sesgo–varianza (4.6).** Los modelos lineales (RL, SVC) y Naive Bayes casi no muestran brecha entre
  entrenamiento y prueba: tienen poca varianza, y su error es sobre todo de **sesgo**. Los árboles profundos,
  Random Forest y la SVM polinomial muestran una brecha mayor porque son más flexibles. En la curva de
  complejidad del árbol, el AUC de entrenamiento sigue subiendo con la profundidad mientras el de validación
  llega a un máximo y cae: ese máximo es el compromiso óptimo. La pérdida del GBT en prueba muestra cuántas
  iteraciones conviene usar.
- **Regularización (3.11 y 4.7).** La validación cruzada elige $\lambda$ y $\alpha$ con datos que el modelo no
  vio, sin tocar la prueba. El camino Lasso muestra cómo $L_1$ anula variables y cuáles sobreviven. Esas
  variables coinciden con las de mayor importancia en los árboles, lo que confirma la selección de forma
  independiente.
- **Naive Bayes (3.8 y 3.9).** Las variantes gaussiana y multinomial parten del mismo supuesto de
  independencia condicional, que aquí no se cumple porque varias variables están correlacionadas. Por eso
  quedan por debajo de los modelos discriminativos. El suavizado de Laplace evita probabilidades nulas, y su
  valor cambia poco el AUC cuando hay muchos datos por categoría.
- **Kernel polinomial (3.10).** Añadir interacciones $x_i x_j$ permite fronteras no lineales. Si la mejora
  frente al SVC lineal es pequeña, la señal es casi lineal en el espacio original. Si es grande, hay
  interacciones relevantes, que es lo que también aprovechan los árboles.
- **Perceptrón vs MLP (4.8).** El perceptrón sin capa oculta queda al nivel de la regresión logística, porque
  ambos son lineales. Añadir capas mejora el ajuste solo hasta cierto punto. Con redes más grandes crece la
  brecha entre entrenamiento y prueba. El descenso de gradiente simple converge más lento que L-BFGS con el
  mismo número de iteraciones.
- **PCA (4.9).** Para conservar el 90 % de la varianza se necesitan varios componentes, y la RL sobre pocos
  componentes pierde AUC frente a usar todas las variables. Esto ilustra que PCA maximiza la varianza y no la
  separación de clases: es una técnica **no supervisada**.
- **Agrupamiento (4.10).** Los clústeres no coinciden con la etiqueta, porque las siluetas son moderadas y
  la clase positiva aparece en todos los grupos. Aun así, la proporción de `>50K` cambia mucho entre
  clústeres, así que el agrupamiento sirve como **segmentación** de la población. La GMM además indica qué
  registros quedan en la frontera entre grupos (probabilidad máxima baja).

## 5. Análisis de Resultados

In [ ]:
### Resumen automático a partir de la tabla de resultados
balanceados = tablaResultados.drop('RL base (sin balanceo)', errors='ignore')
base = resultados['RL base (sin balanceo)']
rlPesos = resultados['Regresion Logistica']

print(f"RL sin balanceo -> Accuracy = {base['Accuracy']:.3f} | Recall >50K = {base['Recall_1']:.3f} | "
      f"Precision >50K = {base['Precision_1']:.3f} | F1 = {base['F1_1']:.3f}")
print(f"RL con pesos    -> Accuracy = {rlPesos['Accuracy']:.3f} | Recall >50K = {rlPesos['Recall_1']:.3f} | "
      f"Precision >50K = {rlPesos['Precision_1']:.3f} | F1 = {rlPesos['F1_1']:.3f}")
print()
for metrica in ['Accuracy', 'Recall_1', 'Precision_1', 'F1_1', 'AUC_ROC', 'AUC_PR']:
    mejor = balanceados[metrica].idxmax()
    print(f"Mejor {metrica:<12}: {mejor:<22} ({balanceados.loc[mejor, metrica]:.3f})")
print()
print("Ranking por AUC-ROC:")
print(balanceados[['AUC_ROC', 'AUC_PR', 'F1_1', 'Recall_1', 'Precision_1', 'Accuracy']].round(3).to_string())

**1. Todos los modelos superan con claridad a la referencia trivial.** Un clasificador que siempre responda
`<=50K` tendría ≈ 76 % de exactitud y un *recall* de 0. Los modelos entrenados alcanzan valores de AUC-ROC muy
por encima de 0.5 y de AUC-PR muy por encima de la prevalencia (≈ 0.24). Esto indica que las variables del
censo sí contienen información suficiente para separar los dos niveles de ingreso.

**2. Efecto del balanceo.** Al comparar la RL sin balanceo con la RL con pesos se ve el intercambio típico. Sin
pesos, el modelo tiene mayor *accuracy* y mayor precisión, pero deja escapar una parte importante de las
personas `>50K` (*recall* moderado). Con pesos, el *recall* sube de forma notable a cambio de más falsos
positivos. Como el desbalance es moderado (24/76), el AUC casi no cambia: el balanceo **mueve el punto de
operación** sobre la misma curva ROC, no mejora la capacidad de ordenar. La decisión depende del uso: si
importa no perder personas de ingreso alto (por ejemplo, para segmentar un mercado), se prefiere el modelo
balanceado. Si importa la exactitud global, se prefiere el modelo sin balanceo o un umbral más alto (ver 4.5).

**3. Comparación entre familias (ver tabla 4.1 y curvas 4.3).**
- Los **ensambles de árboles**, sobre todo **GBT**, suelen obtener el mejor AUC-ROC y AUC-PR en este dataset.
  Capturan interacciones no lineales que un modelo lineal no ve, por ejemplo que la educación pesa distinto
  según el estado civil o la edad, o el efecto de umbral de las ganancias de capital. **Random Forest** queda
  muy cerca.
- La **Regresión Logística** y el **Linear SVC** quedan pocas centésimas de AUC por debajo. Gracias al one-hot
  y a la transformación logarítmica del capital, una frontera lineal ya explica la mayor parte de la señal.
- El **Perceptrón Multicapa** obtiene resultados comparables a la RL, pero cuesta más entrenarlo, necesita
  sobremuestreo y es menos interpretable.
- El **Árbol de Decisión** individual tiene menor AUC, porque da pocas probabilidades distintas (una por
  hoja) y tiene más varianza.
- **Naive Bayes** queda por debajo de los demás. Su supuesto de independencia se rompe claramente: `Sexo`,
  `EstadoCivil` y `Parentesco` están muy correlacionadas, igual que las columnas one-hot de una misma
  variable.

**4. Variables más relevantes (ver 4.4).** Las importancias y los coeficientes coinciden con el análisis
exploratorio: **estado civil casado / parentesco Husband–Wife**, **ganancia de capital**, **años de
educación**, **edad** y **horas por semana** son los factores más asociados a ingresos `>50K`. Le siguen las
ocupaciones directivas y profesionales (`Exec-managerial`, `Prof-specialty`). `Raza` y `PaisOrigen` aportan
poco una vez se incluyen las demás variables. El `Sexo` tiene un peso propio aunque se controlen las demás
variables, lo que refleja la brecha salarial presente en los datos de 1994.

**5. Umbral (ver 4.5).** El umbral que maximiza el F1 suele quedar por encima de 0.5 en los modelos
balanceados, porque los pesos desplazan las probabilidades hacia la clase `>50K`. Ajustar el umbral permite
recuperar precisión sin volver a entrenar.

## 6. Conclusiones

1. **Integración HPC.** Todo el flujo se ejecutó sobre la infraestructura del curso. Los datos se leyeron desde
   **Hadoop HDFS**, y la limpieza, las agregaciones del análisis exploratorio, la codificación, el escalado,
   el entrenamiento y la evaluación se hicieron con **Spark DataFrames y MLlib**. A pandas solo pasaron
   resúmenes y columnas para graficar. El mismo cuaderno escalaría a un censo completo sin cambiar la lógica.

2. **La preparación de datos es determinante.** Convertir los `?` en nulos e imputarlos como `Desconocido`
   conservó todos los registros y una señal útil. Eliminar `PesoMuestral` y `Educacion` (redundante) quitó
   ruido. La transformación $\log(1+x)$ del capital, la agrupación de países y el one-hot dejaron las
   variables en una forma adecuada para todos los modelos.

3. **Se entrenaron y compararon siete modelos de MLlib** (RL, DT, RF, GBT, Linear SVC, MLP y Naive Bayes) con
   la misma partición y las mismas métricas. Para predecir si los ingresos superan los 50K se recomienda el
   modelo con **mayor AUC-ROC y F1 de la clase `>50K`** según la tabla 4.1, que normalmente es
   **Gradient-Boosted Trees**. Si se necesita un modelo explicable, la **Regresión Logística con pesos**
   ofrece un desempeño cercano y coeficientes que se pueden interpretar directamente.

4. **Desbalance y umbral.** Con un desbalance moderado (24/76), los pesos por clase cambian el equilibrio
   entre precisión y *recall*, pero no la capacidad de ordenar del modelo. El umbral de decisión debe elegirse
   según el costo de cada error en la aplicación concreta.

5. **Factores asociados a ingresos altos.** Estar casado, tener más años de educación, tener ganancias de
   capital, tener entre 35 y 60 años, trabajar más de 40 horas y tener una ocupación directiva o profesional.

6. **Consideraciones éticas.** `Sexo` y `Raza` reflejan desigualdades históricas del censo de 1994. Un modelo
   que las use puede reproducir esos sesgos. Si se fuera a usar para decisiones sobre personas, habría que
   evaluar métricas de equidad por grupo o excluir esas variables.

7. **Trabajo futuro.** Ajustar hiperparámetros con `CrossValidator` y `ParamGridBuilder` (sobre todo para GBT
   y RF), probar interacciones explícitas en la RL y validar con la partición original
   `adult.data` / `adult.test` para comparar con resultados publicados (≈ 85–87 % de exactitud).

7. **Temas complementarios del curso.** Los experimentos de la sección 4 conectan los resultados con la teoría
   vista en clase. La validación cruzada y la regularización Elastic Net eligieron la complejidad del modelo
   sin usar la prueba. Las curvas de complejidad y de boosting mostraron el compromiso sesgo–varianza. Lasso
   confirmó las variables más relevantes. PCA mostró que reducir dimensión sin mirar la etiqueta puede perder
   información discriminante. El agrupamiento no supervisado permitió segmentar la población según su
   proporción de casos positivos.

### Cierre de la sesión de Spark

Se liberan los DataFrames que estaban en caché y se detiene la sesión. Así se devuelven al clúster los
executors y la memoria asignados a esta aplicación, y el puerto de la Spark UI queda libre para otros
usuarios.

In [ ]:
### Se liberan los DataFrames en caché y se cierra la sesión de Spark
sparkAvila.catalog.clearCache()
sparkAvila.stop()
print("Sesión de Spark finalizada")